In [9]:
import numpy as np
import pandas as pd

In [11]:
# Load the datasets
try:
    customer_info = pd.read_csv('customer_info.csv')
    purchase_data = pd.read_csv('purchase_data.csv')
    print("Datasets loaded successfully.")
except FileNotFoundError as e:
    print(f"Error loading files: {e}")

Datasets loaded successfully.


In [47]:
# Handle Missing Values
median_age = customer_info["Age"].median()
customer_info["Age"] = customer_info["Age"].fillna(median_age)
print(customer_info["Age"].isnull().sum())

0


In [15]:
# Remove Duplicates
customer_info = customer_info.drop_duplicates()
print(customer_info.shape)

(49, 4)


In [25]:
# calculate IQR
Q1 = purchase_data["PurchaseAmount"].quantile(0.25)
Q3 = purchase_data["PurchaseAmount"].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR
print("Q1 is", Q1)
print("Q3 is", Q3)
print("The IQR calculated is", IQR)
print("Lower Bound is", lower_bound)
print("Upper Bound is", upper_bound)
print("="*50)

purchase_data_cleaned = purchase_data[
    (purchase_data["PurchaseAmount"] >= lower_bound) &
    (purchase_data["PurchaseAmount"] <= upper_bound)
]
print("Cleaned Data:",purchase_data_cleaned.head(5))
print("="*50)
outliers = purchase_data[
    (purchase_data["PurchaseAmount"] < lower_bound) |
    (purchase_data["PurchaseAmount"] > upper_bound)
]
print("Outliers Data:",outliers)

Q1 is 90.0
Q3 is 307.5
The IQR calculated is 217.5
Lower Bound is -236.25
Upper Bound is 633.75
Cleaned Data:    Cust_ID  PurchaseAmount PurchaseDate
0      101           150.0   2023-01-01
1      102           200.5   2023-01-02
2      103            50.0   2023-01-03
4      106            75.0   2023-01-05
5      107           300.0   2023-01-06
Outliers Data:     Cust_ID  PurchaseAmount PurchaseDate
3       104         10000.0   2023-01-04
31      133           720.0   2023-02-01
37      139          1500.0   2023-02-07
45      147           800.0   2023-02-15
48      150          9999.0   2023-02-18


In [27]:
# Data Integration
combined_data = pd.merge(
    customer_info,
    purchase_data_cleaned,
    left_on="CustomerID",
    right_on="Cust_ID",
    how="inner"
)
print(combined_data.head(5))

   CustomerID     Name   Age         City  Cust_ID  PurchaseAmount  \
0         101    Alice  25.0     New York      101           150.0   
1         102      Bob  30.0  Los Angeles      102           200.5   
2         103  Charlie  33.0      Chicago      103            50.0   
3         106    Frank  35.0      Houston      106            75.0   
4         107    Grace  42.0       Boston      107           300.0   

  PurchaseDate  
0   2023-01-01  
1   2023-01-02  
2   2023-01-03  
3   2023-01-05  
4   2023-01-06  


In [29]:
# Data Transformation
# Data Type Conversion
print("before",combined_data.dtypes)
print("="*50)
combined_data["PurchaseDate"] = pd.to_datetime(combined_data["PurchaseDate"])
print("after",combined_data.dtypes)

before CustomerID          int64
Name                  str
Age               float64
City                  str
Cust_ID             int64
PurchaseAmount    float64
PurchaseDate          str
dtype: object
after CustomerID                 int64
Name                         str
Age                      float64
City                         str
Cust_ID                    int64
PurchaseAmount           float64
PurchaseDate      datetime64[us]
dtype: object


In [33]:
# Feature Engineering
combined_data["HighSpender"] = combined_data["PurchaseAmount"] > 150
print(combined_data.head(5))

   CustomerID     Name   Age         City  Cust_ID  PurchaseAmount  \
0         101    Alice  25.0     New York      101           150.0   
1         102      Bob  30.0  Los Angeles      102           200.5   
2         103  Charlie  33.0      Chicago      103            50.0   
3         106    Frank  35.0      Houston      106            75.0   
4         107    Grace  42.0       Boston      107           300.0   

  PurchaseDate  HighSpender  
0   2023-01-01        False  
1   2023-01-02         True  
2   2023-01-03        False  
3   2023-01-05        False  
4   2023-01-06         True  


In [35]:
# Dimensionality Reduction
combined_data = combined_data.drop(columns=["Name"])
print(combined_data.head(5))

   CustomerID   Age         City  Cust_ID  PurchaseAmount PurchaseDate  \
0         101  25.0     New York      101           150.0   2023-01-01   
1         102  30.0  Los Angeles      102           200.5   2023-01-02   
2         103  33.0      Chicago      103            50.0   2023-01-03   
3         106  35.0      Houston      106            75.0   2023-01-05   
4         107  42.0       Boston      107           300.0   2023-01-06   

   HighSpender  
0        False  
1         True  
2        False  
3        False  
4         True  


In [39]:
# Sampling
sample_data = combined_data.sample(frac=0.6, random_state=42)
print(sample_data.head(5))

    CustomerID   Age      City  Cust_ID  PurchaseAmount PurchaseDate  \
37         142  43.0    Dallas      142            90.0   2023-02-10   
24         127  39.0    Dallas      127           340.0   2023-01-26   
25         128  44.0  New York      128           210.0   2023-01-27   
36         141  35.0     Miami      141           275.0   2023-02-09   
34         138  31.0   Houston      138           340.0   2023-02-06   

    HighSpender  
37        False  
24         True  
25         True  
36         True  
34         True  


In [41]:
# Data Discretization
bins = [18, 30, 50, float("inf")]
labels = ["Young", "Middle-Aged", "Senior"]

combined_data["AgeGroup"] = pd.cut(
    combined_data["Age"],
    bins=bins,
    labels=labels,
    right=False 
)
print(combined_data.head(10))

   CustomerID   Age         City  Cust_ID  PurchaseAmount PurchaseDate  \
0         101  25.0     New York      101           150.0   2023-01-01   
1         102  30.0  Los Angeles      102           200.5   2023-01-02   
2         103  33.0      Chicago      103            50.0   2023-01-03   
3         106  35.0      Houston      106            75.0   2023-01-05   
4         107  42.0       Boston      107           300.0   2023-01-06   
5         108  27.0      Chicago      108           120.0   2023-01-07   
6         109  31.0      Seattle      109           180.0   2023-01-08   
7         110  29.0        Miami      110            90.0   2023-01-09   
8         111  33.0       Dallas      111           250.0   2023-01-10   
9         112  26.0     New York      112            60.0   2023-01-11   

   HighSpender     AgeGroup  
0        False        Young  
1         True  Middle-Aged  
2        False  Middle-Aged  
3        False  Middle-Aged  
4         True  Middle-Aged  
5    